# D11：按资产分组与聚合

学习日期：2026-10-25。

目标：统计记录数、非空价格数、起止日期、最高价和最低价；
用 agg 整理为一张表，再用手算 A 的结果核对。

先打开 practice.ipynb 独立写，遇到困难再查本参考。
右上角内核选择 Python (D Learning)，按从上到下的顺序运行。
Shift+Enter 运行当前单元格；Ctrl+S 保存。

本次基于 D9 的原始八行建立独立教学副本，仅将编号 105 的价格设为空。
D10 清洗后 A 只有三行，而本次 A 保留四行，统计结果不能混用。
缺失价格用于比较 size 和 count；今天不删除这条记录，也不把缺失填成 0。


## 1. 读取 D11 的练习样本

本次只使用 `data/prices.csv`。它有 8 行、5 列，A 与 B 各有四条记录。
这是 D9 原始八行样本的教学副本，只把编号 105 的价格设为空。
`data/prices_original.csv` 保留原来的八行，供追溯；D9、D10 的文件没有被修改。

本例使用 `read_csv` 默认设置，CSV 中的空字段会识别为缺失值。
不要沿用 D10 的 `keep_default_na=False`：D10 当时需要保留原始文字，今天直接统计非空数。
代码采用 D11 的绝对路径，因此终端当前处于 `D:/Learning` 或项目根目录都能找到样本。
运行前预测：`df.shape` 应为 `(8, 5)`，编号 105 的价格应显示为 `NaN`。


In [ ]:
from pathlib import Path  # Path：用于表示、拼接文件和文件夹路径
import pandas as pd  # pandas：表格处理库；pd 是常用简称

base_dir = Path("D:/Learning/study/d11")  # 本次 D11 文件夹的绝对路径
data_path = base_dir / "data" / "prices.csv"  # / 在 Path 之间用于拼接路径
output_dir = base_dir / "outputs"  # 之后保存汇总表的文件夹

df = pd.read_csv(data_path)  # 读取 CSV 为 DataFrame；空价格字段自动识别为缺失
print(df)  # 查看全部八条记录；编号 105 的 price 应为 NaN
print("原表形状：", df.shape)  # shape 是属性，预计输出 (8, 5)


## 2. 确认价格和日期的类型

统计最高价、最低价前，让价格成为数值；统计起止日期前，让日期成为日期类型。
`errors="coerce"` 表示无法转换的内容变成缺失值，不能理解成已经修复原数据。
这份教学样本只有一条空价格，没有非法日期、重复记录或价格冲突。

运行前预测：`price` 缺失数为 1，其他列缺失数为 0。
今天的“有效价格数”指完成数值转换后，价格列的非空数量。
`count()` 本身不检查价格是否符合业务规则。


In [ ]:
df["price"] = pd.to_numeric(  # 将价格列明确转换为数值类型
    df["price"],  # 指定需要转换的原始价格列
    errors="coerce",  # 无法转成数字的值转为缺失值；已有缺失值仍然缺失
)  # 将转换结果放回 price 列

df["date"] = pd.to_datetime(  # 将日期文字转换为日期类型
    df["date"],  # 指定需要转换的原始日期列
    format="%Y-%m-%d",  # %Y 年、%m 月、%d 日；例如 2026-10-12
    errors="coerce",  # 非法日期转换为 NaT，即日期缺失值
)  # 将转换结果放回 date 列

df.info()  # 查看列、非空数量和类型；直接调用即可，不需要外面再套 print
print(df.isna().sum())  # isna 标记缺失，再逐列计数；预计 price 为 1，其他为 0


## 3. 按资产分组，观察 A 组

`grouped` 是自定义变量名，保存 GroupBy 分组对象。
`groupby("asset")` 根据 asset 列的值分组，A 的所有记录归为一组，B 的所有记录归为另一组。
单独分组还没有生成汇总表；原始八行也没有消失。

`ngroups` 表示组数，是属性，因此后面没有括号。
`get_group("A")` 取出 A 组的全部原始记录，是方法，因此需要括号。
运行前预测：有 2 组；A 组形状为 `(4, 5)`；A 的价格为 100、105、缺失、108。


In [ ]:
grouped = df.groupby("asset")  # 依据 asset 列分组，得到 GroupBy 对象
print("资产组数：", grouped.ngroups)  # ngroups 是组数，预计输出 2

a_records = grouped.get_group("A")  # 提取 asset 等于 A 的四条记录
print(a_records)  # 显示 A 组全部记录，包括价格缺失的那一行
print("A 组形状：", a_records.shape)  # 仍有四行五列，预计为 (4, 5)


## 4. 先分别计算六项统计

`size()` 统计每组的行数，价格缺失的记录仍算一行。
`grouped["price"].count()` 统计每组 price 列的非空值数量。
`grouped["price"]` 表示在每个资产组内，选择价格列继续计算。

运行前预测：A 的记录数 4、非空价格数 3；B 的两项都是 4。
价格的 min/max 忽略缺失值。这里日期范围覆盖各资产的全部记录，未先删掉空价格行。
本例日期完整；若整组价格都缺失，最高价与最低价也会缺失，不应改成 0。


In [ ]:
record_counts = grouped.size()  # 统计每个资产组的记录行数，含空价格行
valid_price_counts = grouped["price"].count()  # 统计每组非空价格数
start_dates = grouped["date"].min()  # 每组最早日期；不需要先按日期排序
end_dates = grouped["date"].max()  # 每组最晚日期
highest_prices = grouped["price"].max()  # 每组最高价，忽略价格缺失值
lowest_prices = grouped["price"].min()  # 每组最低价，忽略价格缺失值

print("记录数：", record_counts)  # A 为 4，B 为 4
print("非空价格数：", valid_price_counts)  # A 为 3，B 为 4
print("起始日期：", start_dates)  # A、B 都是 2026-10-12
print("结束日期：", end_dates)  # A、B 都是 2026-10-15
print("最高价：", highest_prices)  # A 为 108，B 为 205
print("最低价：", lowest_prices)  # A 为 100，B 为 198


## 5. 用 agg 整理成一张汇总表

`agg` 是 aggregate 的简写，意为聚合、汇总。
`as_index=False` 让分组键 asset 保留为结果中的普通列。
下面使用“命名聚合”：`结果列名=(原表列名, 统计方法)`。

例如 `valid_price_count=("price", "count")`：
`valid_price_count` 是自定义的结果列名；`price` 必须是原表已有的列；`count` 是统计方法。
括号中的两个值组成一个元组。
`record_count=("price", "size")` 使用 size，因此价格缺失的那行也计入。

运行前预测：A、B 各输出一行；asset 加六项统计，共 `(2, 7)`。


In [ ]:
grouped_for_summary = df.groupby(  # 创建用于整理汇总表的分组对象
    "asset",  # 仍根据资产编号分组
    as_index=False,  # 分组键 asset 保持为普通列，结果使用默认数字索引
)  # 这里尚未执行统计

summary = grouped_for_summary.agg(  # agg：一次完成并命名多项统计
    record_count=("price", "size"),  # 每组记录数，包含空价格行
    valid_price_count=("price", "count"),  # 每组非空价格数
    start_date=("date", "min"),  # 每组最早日期，覆盖全部记录
    end_date=("date", "max"),  # 每组最晚日期，覆盖全部记录
    highest_price=("price", "max"),  # 每组最高价
    lowest_price=("price", "min"),  # 每组最低价
)  # 每组整理成一行；结果保存到 summary

print(summary)  # 查看 A、B 两行汇总结果
print("汇总表形状：", summary.shape)  # 预计输出 (2, 7)


## 6. 手算 A，并逐项核对

先在纸上写答案，再运行本节。不要把程序输出直接抄成“手算结果”。

A 的四条记录日期是 10 月 12、13、14、15 日，价格是 100、105、缺失、108。
所以记录数为 4，非空价格数为 3，最高价为 108，最低价为 100；
起止日期为 2026-10-12 和 2026-10-15。
即使 10 月 14 日价格缺失，那条记录仍属于 A 组。

以下字典保存独立手写的预期值。
`pd.Timestamp` 用于把一个日期文字表示为单个 Pandas 日期值，方便与统计结果比较。
临时用 set_index 把 asset 作为索引，再 loc["A"] 取到 A 的一行统计，不修改原 summary。
这次取单行得到 Series，其索引是六项统计字段。


In [ ]:
manual_a = {  # 字典：保存从纸上独立手算得到的 A 组答案
    "record_count": 4,  # 四条记录，其中一条价格缺失
    "valid_price_count": 3,  # 只有三个价格不是缺失值
    "start_date": pd.Timestamp("2026-10-12"),  # 最早日期
    "end_date": pd.Timestamp("2026-10-15"),  # 最晚日期
    "highest_price": 108,  # 非空价格 100、105、108 中的最大值
    "lowest_price": 100,  # 非空价格 100、105、108 中的最小值
}  # 字典创建完成

summary_by_asset = summary.set_index("asset")  # 返回以资产为索引的新表
a_summary = summary_by_asset.loc["A"]  # 按标签 A 取出该行，得到一个 Series

for field, expected in manual_a.items():  # 每次取出一个字段名和对应手算值
    actual = a_summary[field]  # 按字段名读取程序统计出的实际值
    print(field, "程序：", actual, "手算：", expected)  # 并列查看两种结果
    assert actual == expected  # assert：实际值不相等时停止并报错，帮助发现问题

print("A 组的六项统计与手算一致。")  # 前面六项全部相等才会执行到这里


## 7. 导出汇总表，并检查总数

保存到 `outputs/asset_summary.csv`。重复运行会更新这个结果文件。
`index=False` 表示不把表左侧的默认数字索引另写成一列，asset 作为普通列仍会保存。
`utf-8-sig` 是带标记的 UTF-8 编码，方便 Windows 软件识别中文。
`date_format` 控制导出的日期写法。CSV 保存文字和数值内容，不保留 Pandas 的列类型。

运行前预测：记录数合计 8，非空价格数合计 7。
汇总表只有两行，因为有 A、B 两个资产组，每组的六项统计聚合成一行。
不是只保留了两条原始记录，原始 df 仍然有八行。


In [ ]:
output_dir.mkdir(  # 创建保存结果的文件夹
    parents=True,  # 上级文件夹不存在时一并创建
    exist_ok=True,  # 文件夹已经存在时继续执行
)  # 此时输出文件夹已准备好

summary_path = output_dir / "asset_summary.csv"  # 拼接结果文件的完整路径
summary.to_csv(  # 将汇总表写成 CSV 文件
    summary_path,  # 输出文件路径
    index=False,  # 不输出额外的默认索引列；asset 普通列仍保留
    encoding="utf-8-sig",  # 采用带标记的 UTF-8 编码
    date_format="%Y-%m-%d",  # 日期输出为例如 2026-10-12
)  # CSV 保存完成

print("结果保存位置：", summary_path)  # 显示刚保存的结果路径
print("记录数合计：", summary["record_count"].sum())  # 预计为 8
print("非空价格数合计：", summary["valid_price_count"].sum())  # 预计为 7
print("原表仍有多少行：", len(df))  # 原表仍为八行，聚合不会删除原表记录


## 8. 独立验收

1. 在 practice.ipynb 的空白单元格中，不看答案重写一次 agg。
2. 用自己的话解释：为什么输出只有两行？这两行各代表什么？
3. 解释 A 为什么 record_count=4、valid_price_count=3。
4. 打开 outputs/asset_summary.csv，确认列名包含 asset，且没有 Unnamed: 0 一列。
5. 回答 notes.md；完成后再查 notes_reference.md。

选做：对 summary.copy() 新增 missing_price_count 列，计算记录数减去非空价格数。
预计 A 为 1、B 为 0；只在练习副本中操作，主汇总仍保留七列。
